In [1]:
%load_ext autoreload
%autoreload 2

In [20]:
import pandas as pd

In [51]:
import esp_data.file_io.functional as F

In [5]:
from esp_data.file_io.utils import make_fs

In [9]:
from esp_data.paths import AnyPath

# Testing the fileio suite

In [7]:
gfs = make_fs("gs://")

In [8]:
type(gfs)

gcsfs.core.GCSFileSystem

## Test file open

In [11]:
import csv
from io import StringIO

In [24]:
file_path = "gs://esp-ci-cd-tests/esp-data-tests/test_csv.csv"

In [18]:
def write_rows_to_csv(
    rows: list[dict], *, path: str | AnyPath, mode: str = "a", fs=None, write_header: bool = False
) -> None:
    fieldnames = list(rows[0].keys())
    # Write batch to string buffer
    output = StringIO()
    writer = csv.DictWriter(output, fieldnames=fieldnames)
    if write_header:
        writer.writeheader()

    writer.writerows(rows)

    if fs is None:
        with AnyPath(path).open(mode=mode) as f:
            f.write(output.getvalue())
    else:
        with fs.open(path, mode=mode) as f:
            f.write(output.getvalue())

In [13]:
rows1 = [
    {"col1": 1, "col2": "a"},
    {"col1": 2, "col2": "b"},
]

rows2 = [
    {"col1": 3, "col2": "c"},
    {"col1": 4, "col2": "d"},
]

In [38]:
write_rows_to_csv(rows1, path=file_path, mode="w", fs=gfs, write_header=True)

In [39]:
df = pd.read_csv(file_path)
df

,col1,col2
0,1,a
1,2,b


In [40]:
write_rows_to_csv(rows2, path=file_path, mode="a", fs=gfs, write_header=False)

In [41]:
df2 = pd.read_csv(file_path)
df2

,3,c
0,4,d


In [42]:
F.delete_file(file_path)

True

In [43]:
F.exists(file_path)

False

## looks like open with mode="a" fails in gcsfs
Let's try with AnyPath

In [44]:
write_rows_to_csv(rows1, path=file_path, mode="w", fs=None, write_header=True)

In [45]:
df = pd.read_csv(file_path)
df

,col1,col2
0,1,a
1,2,b


In [46]:
write_rows_to_csv(rows2, path=file_path, mode="a", fs=None, write_header=False)

In [47]:
df2 = pd.read_csv(file_path)
df2

,col1,col2
0,1,a
1,2,b
2,3,c
3,4,d


## test makedirs

In [52]:
F.makedirs("gs://esp-ci-cd-tests/esp-data-tests/dir_tests/")

OSError: Forbidden: https://storage.googleapis.com/upload/storage/v1/b/tmp/o
gagan@earthspecies.org does not have storage.objects.create access to the Google Cloud Storage object. Permission 'storage.objects.create' denied on resource (or it may not exist).